# Task 3 — Colab notebook for the test-set creator

Этот notebook нужен **эксперту-создателю набора**. Он делает два артефакта:

1. минимальный trajectory YAML/ZIP для Task 3;
2. authoring bundle с offline HTML и JSON-шаблоном, где описываются curated hard cases.

## Что получится на выходе

- `task1_minimal_for_ab_bundle.zip`
- `task3_ab_creator_bundle.zip`
- `task3_ab_case_manifest.template.json`
- `task3_ab_creator_offline_form.html`

Дальше создатель открывает offline HTML, заполняет кейсы и экспортирует `task3_ab_case_manifest.filled.json`. Именно этот JSON потом идёт в CLI runner.

In [ ]:
# @title Install + bootstrap repo
!pip -q install pyyaml ipywidgets nbformat

import datetime as dt
import hashlib
import json
import re
import sys
import unicodedata
import zipfile
from pathlib import Path

import yaml

REPO_DIR = Path('/content/top-papers-graph')
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

from scireason.task3_ab_case_manifest import default_case_manifest, build_task3_ab_authoring_bundle

CURRENT_YEAR = dt.datetime.utcnow().year

def _slugify(text: str) -> str:
    text = str(text or '').strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9а-яё]+', '-', text, flags=re.IGNORECASE)
    text = re.sub(r'-{2,}', '-', text).strip('-')
    return text or 'submission'


def _short_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode('utf-8')).hexdigest()[:10]


def build_minimal_task1_yaml(*, topic: str, cutoff_year: int, identifiers_text: str, expert_name: str = '') -> dict:
    papers = []
    for raw in re.split(r'[
;,]+', identifiers_text or ''):
        raw = raw.strip()
        if not raw:
            continue
        papers.append({'id': raw, 'title': '', 'year': 0})
    submission_seed = topic or identifiers_text or 'task3-ab'
    expert_name = (expert_name or '').strip()
    return {
        'submission_id': f"{_slugify(expert_name or 'expert')}__{_short_hash(submission_seed)}",
        'topic': topic.strip(),
        'cutoff_year': int(cutoff_year),
        'domain': 'science',
        'expert': {'full_name': expert_name, 'latin_slug': _slugify(expert_name or 'expert')},
        'papers': papers,
        'steps': [],
        'edges': [],
    }


def write_task1_bundle(payload: dict, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    yaml_path = out_dir / 'task1_minimal_for_ab.yaml'
    yaml_path.write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding='utf-8')
    (out_dir / 'manifest.json').write_text(json.dumps({'submission_id': payload.get('submission_id'), 'topic': payload.get('topic')}, ensure_ascii=False, indent=2), encoding='utf-8')
    (out_dir / 'README.txt').write_text('Upload task1_minimal_for_ab.yaml or the ZIP bundle into the Task 3 runner.
', encoding='utf-8')
    zip_path = out_dir / 'task1_minimal_for_ab_bundle.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [yaml_path, out_dir/'manifest.json', out_dir/'README.txt']:
            zf.write(p, arcname=p.name)
    return zip_path


In [ ]:
# @title Fill these fields
EXPERT_NAME = ''
TOPIC = ''
CUTOFF_YEAR = CURRENT_YEAR
IDENTIFIERS_TEXT = '''
'''
DEFAULT_CASE_COUNT = 20
OUTPUT_DIR = Path('/content/task3_ab_creator_outputs')


In [ ]:
# @title Build trajectory bundle + creator bundle
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trajectory_payload = build_minimal_task1_yaml(topic=TOPIC, cutoff_year=CUTOFF_YEAR, identifiers_text=IDENTIFIERS_TEXT, expert_name=EXPERT_NAME)
trajectory_zip = write_task1_bundle(trajectory_payload, OUTPUT_DIR / 'trajectory_bundle')
seed_manifest = default_case_manifest(topic=TOPIC, submission_id=trajectory_payload['submission_id'], creator_id=EXPERT_NAME, cutoff_year=str(CUTOFF_YEAR), case_count=DEFAULT_CASE_COUNT)
authoring_assets = build_task3_ab_authoring_bundle(output_dir=OUTPUT_DIR / 'authoring_bundle', seed_manifest=seed_manifest)

print('Trajectory ZIP:', trajectory_zip)
for key, value in authoring_assets.items():
    print(f'{key}: {value}')


## Что делать дальше

1. Скачайте `task1_minimal_for_ab_bundle.zip`.
2. Скачайте `task3_ab_creator_bundle.zip` или `task3_ab_creator_offline_form.html`.
3. Откройте offline HTML локально, заполните кейсы и экспортируйте `task3_ab_case_manifest.filled.json`.
4. Передайте trajectory ZIP, filled JSON и curated `processed_papers` в CLI/Kaggle/DataSphere runner.